# 03 · Clustering: agglomerative and k-medoids

**Lecture 2 · Block B6 · Lab · 25 minutes (if time is short: one of Task 1 / Task 2 in class, the other at home)**

**Learning goal:** We implement one hierarchical algorithm (agglomerative clustering) and one partitioning algorithm (k-medoids) on our own distance matrix, read a dendrogram, and compare our results with SciPy and scikit-learn.

**What we do:**
1. Task 1 - `agglomerative(D, linkage)` (core, about 10 min)
2. Task 2 - `k_medoids(D, k, seed)` (core, about 10 min)
3. Task 3 - optional alternative: `k_means(X, k, distance, seed)` with our own distance

k-medoids is the easier one: two short steps in a loop. In agglomerative clustering the tricky part is the bookkeeping (which row holds which cluster); that part is given, we write the two key lines. The PCA/MDS plots at the end are prepared.

In [ ]:
DATASET = "penguins"  # or "wholesale", "auto_mpg", "students" (see 01_dataset.ipynb)

In [ ]:
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.cluster.hierarchy import fcluster
from scipy.cluster.hierarchy import linkage as scipy_linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score

import ddm

## Setup: load X and D from 02_distance (given)

In [ ]:
df = ddm.load_dataset(DATASET)
info = ddm.dataset_info(DATASET)

X = ddm.load_array(DATASET, "X")  # feature matrix saved by 02_distance.ipynb
D = ddm.load_array(DATASET, "D")  # distance matrix saved by 02_distance.ipynb
meta = ddm.load_json(DATASET, "meta")  # which rows, which types, ...

if X is None or D is None or meta is None:
    missing = [name for name, obj in [("X", X), ("D", D), ("meta", meta)] if obj is None]
    print(f"NOTICE: results of lecture01/02_distance.ipynb not found in {ddm.processed_dir()} (missing: {', '.join(missing)}).")
    print("        Using a BASELINE instead: z-scored numeric columns + Euclidean distance (scipy pdist).")
    print("        Solve lecture01/02_distance.ipynb and run it to the end to work with your own X and D.")
    numeric_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c]) and c not in info.label_candidates
    ]
    base = df[numeric_cols].dropna()
    X = ((base - base.mean()) / base.std()).to_numpy()
    D = squareform(pdist(X, metric="euclidean"))
    meta = {
        "rows": base.index.tolist(),
        "types": {c: "numeric" for c in numeric_cols},
        "log_columns": [],
        "feature_names": numeric_cols,
        "source": "baseline",
    }
else:
    print(f"Loaded X {X.shape} and D {D.shape} from {ddm.processed_dir()} (saved by 02_distance.ipynb).")

data = df.loc[meta["rows"]]  # the objects (rows) that X and D describe, in the same order
TYPES = meta["types"]
print(f"{len(data)} objects, {X.shape[1]} features in X.")

In [ ]:
K = 3  # number of clusters we ask for (penguins: 3 species). Change it for your dataset.
LABEL = info.label_candidates[0]  # known groups: only used for colouring and comparing
truth = data[LABEL].fillna("missing").astype(str).to_numpy()
print(f"K = {K}, known groups = '{LABEL}' with values {sorted(set(truth))}")

## Task 1 · Agglomerative clustering (core, about 10 minutes)

Start with every object as its own cluster. Repeat $n - 1$ times: **merge the two closest
clusters**, then **compute the distance of the new cluster to all others**. For the second
step we only need the old distances (Lance–Williams update), with $|A|$ = number of objects in $A$:

| Linkage | $d(A \cup B,\ C)$ | Character |
|---------|-------------------|-----------|
| single | $\min\big(d(A, C),\ d(B, C)\big)$ | chains, long thin clusters |
| complete | $\max\big(d(A, C),\ d(B, C)\big)$ | compact clusters |
| average | $\dfrac{|A|\, d(A, C) + |B|\, d(B, C)}{|A| + |B|}$ | in between |

**Output format (SciPy):** a matrix `Z` with $n - 1$ rows `[id_a, id_b, distance, size]`.
Objects have the ids $0 \dots n-1$; the cluster built in step `step` gets the id `n + step`.
With this format, SciPy can draw the dendrogram and cut the tree for us.

**Our part:** step 1 (find the closest pair) and step 3 (one line per linkage, store it in `new`).
The rest is given.

<details>
<summary><b>Hint</b> (click to open)</summary>

- `np.argmin(dist)` returns the position of the smallest entry in the *flattened* matrix;
  `np.unravel_index(position, dist.shape)` turns it into a pair `(a, b)`.
- `dist[a]` is the whole row of cluster `a`: the distances to all other clusters.
  `np.minimum(dist[a], dist[b])` works element-wise.
- Switched-off rows contain `np.inf`, so they never win the `argmin` and do not disturb `min`/`max`/average.
</details>

In [ ]:
def agglomerative(D: np.ndarray, linkage: str = "average") -> np.ndarray:
    """Agglomerative (bottom-up) hierarchical clustering on a distance matrix.

    Parameters
    ----------
    D : array of shape (n, n)
        Symmetric distance matrix with zero diagonal.
    linkage : "single", "complete" or "average"

    Returns
    -------
    Z : array of shape (n - 1, 4), SciPy linkage format
        Row ``step`` = ``[id_a, id_b, distance, size]`` of the merge in that
        step. Objects have the ids ``0 .. n-1``; the cluster created in step
        ``step`` gets the id ``n + step``.
    """
    D = np.asarray(D, dtype=float)
    n = D.shape[0]
    dist = D.copy()  # dist[a, b] = distance between the clusters stored in rows a and b
    np.fill_diagonal(dist, np.inf)  # a cluster is never merged with itself
    cluster_id = np.arange(n)  # SciPy id of the cluster stored in each row
    size = np.ones(n, dtype=int)  # number of objects in the cluster stored in each row
    Z = np.zeros((n - 1, 4))

    for step in range(n - 1):
        # 1. TODO: find the two closest clusters: row a and column b of the smallest entry of dist
        raise NotImplementedError("agglomerative: step 1, find the closest pair")
        a, b = min(a, b), max(a, b)

        # 2. write the merge into Z (given)
        id_a, id_b = sorted((cluster_id[a], cluster_id[b]))
        Z[step] = [id_a, id_b, dist[a, b], size[a] + size[b]]

        # 3. TODO: distance of the merged cluster (a + b) to every other cluster -> store it in `new`
        if linkage == "single":
            raise NotImplementedError("agglomerative: step 3, single linkage")
        elif linkage == "complete":
            raise NotImplementedError("agglomerative: step 3, complete linkage")
        elif linkage == "average":
            raise NotImplementedError("agglomerative: step 3, average linkage")
        else:
            raise ValueError(f"Unknown linkage '{linkage}'")

        # 4. bookkeeping (given): the merged cluster lives on in row a, row b is switched off
        dist[a, :] = new
        dist[:, a] = new
        dist[a, a] = np.inf
        dist[b, :] = np.inf
        dist[:, b] = np.inf
        size[a] = size[a] + size[b]
        cluster_id[a] = n + step
    return Z

In [ ]:
# Check Task 1: a small example by hand, then the full D against scipy.cluster.hierarchy.linkage
points = np.array([0.0, 1.0, 3.0, 7.0])  # four objects on a line
D_small = np.abs(points[:, None] - points[None, :])
EXPECTED_HEIGHTS = {"single": [1, 2, 4], "complete": [1, 3, 7], "average": [1, 2.5, 17 / 3]}


def make_test(method: str):
    def test():
        Z_small = np.asarray(agglomerative(D_small, method))
        assert Z_small.shape == (3, 4), f"Z must have n-1 = 3 rows and 4 columns, got {Z_small.shape}"
        assert np.allclose(Z_small[:, 2], EXPECTED_HEIGHTS[method]), \
            f"small example: merge heights {Z_small[:, 2]}, expected {EXPECTED_HEIGHTS[method]}"
        assert Z_small[-1, 3] == 4, "the last merge must contain all 4 objects"
        Z = np.asarray(agglomerative(D, method))
        Z_ref = scipy_linkage(squareform(D, checks=False), method=method)
        assert np.allclose(np.sort(Z[:, 2]), np.sort(Z_ref[:, 2])), "our data: merge heights differ from scipy"
        ari = adjusted_rand_score(fcluster(Z, K, "maxclust"), fcluster(Z_ref, K, "maxclust"))
        assert ari > 0.999, f"our data: the cut into {K} clusters differs from scipy (ARI {ari:.3f})"
    return test


for method in ("single", "complete", "average"):
    ddm.check(f"agglomerative, {method} linkage", make_test(method))

### Dendrogram and cut (given)

We draw the last 40 merges of the tree and cut it at a height that leaves `K` clusters.
If our function is not finished yet, SciPy steps in (with a notice).

In [ ]:
LINKAGE = "average"

try:
    Z = agglomerative(D, LINKAGE)
    source = "our agglomerative"
except NotImplementedError:
    Z = scipy_linkage(squareform(D, checks=False), method=LINKAGE)
    source = "SciPy - NOTICE: our agglomerative is not implemented yet"

cut = (Z[-K, 2] + Z[-K + 1, 2]) / 2  # between the merges that leave K and K-1 clusters
ddm.plot_dendrogram(Z, cut_height=cut, truncate_mode="lastp", p=40,
                    title=f"{LINKAGE} linkage, {DATASET} ({source})")
plt.show()

labels_agg = fcluster(Z, K, criterion="maxclust") - 1  # SciPy counts from 1
print(pd.crosstab(labels_agg, truth, rownames=["cluster"], colnames=[LABEL]))

### Three linkages on the same D (given)

In [ ]:
linkage_labels = {}
for method in ("single", "complete", "average"):
    try:
        Z_m = agglomerative(D, method)
        name = f"{method} linkage (ours)"
    except NotImplementedError:
        Z_m = scipy_linkage(squareform(D, checks=False), method=method)
        name = f"{method} linkage (SciPy)"
    linkage_labels[name] = fcluster(Z_m, K, criterion="maxclust") - 1
    sizes = np.bincount(linkage_labels[name])
    ari = adjusted_rand_score(truth, linkage_labels[name])
    print(f"{name:26s} cluster sizes {sizes.tolist()}  ARI vs {LABEL}: {ari:.3f}")

ddm.plot_clusters_mds(D, {f"{LABEL} (known groups)": truth, **linkage_labels},
                      title=f"{DATASET}: same distance matrix, three linkages (MDS projection)")
plt.show()

## Task 2 · k-medoids (core, about 10 minutes)

A *medoid* is a real object that represents its cluster. k-medoids works with **any**
distance matrix. Objective: make the total distance of all objects to their medoid small,

$$\text{cost} = \sum_{i=1}^{n} d\big(i,\ m_{c(i)}\big).$$

Algorithm (alternating version):

1. Start with `k` random objects as medoids (given).
2. **Assignment:** every object goes to its closest medoid.
3. **Update:** in every cluster, the new medoid is the member with the smallest sum of
   distances to all other members.
4. Repeat 2-3 until the medoids do not change (given).

<details>
<summary><b>Hint</b> (click to open)</summary>

- `D[:, medoids]` is an $n \times k$ matrix: distance of every object to every medoid.
  `np.argmin(..., axis=1)` gives the closest medoid per object.
- `D[np.ix_(members, members)]` is the distance matrix *within* one cluster;
  `.sum(axis=1)` gives the sum of distances for every member.
- `members[np.argmin(...)]` translates the position inside the cluster back to an object index.
</details>

In [ ]:
def k_medoids(D: np.ndarray, k: int, seed: int = 0, max_iter: int = 100):
    """k-medoids clustering (alternating algorithm) on a distance matrix.

    Parameters
    ----------
    D : array of shape (n, n)
        Distance matrix (any distance: Gower, Euclidean, ...).
    k : number of clusters
    seed : seed for the random start medoids
    max_iter : maximum number of assignment/update rounds

    Returns
    -------
    labels : int array of shape (n,), values 0 .. k-1
    medoids : int array of shape (k,); ``medoids[c]`` is the row index of the
        object that represents cluster c
    """
    D = np.asarray(D, dtype=float)
    n = D.shape[0]
    rng = np.random.default_rng(seed)
    medoids = rng.choice(n, size=k, replace=False)  # start: k random objects

    for _ in range(max_iter):
        # TODO 1 (assignment): labels = index 0..k-1 of the closest medoid, for every object
        raise NotImplementedError("k_medoids: assignment step")

        new_medoids = medoids.copy()
        for c in range(k):
            members = np.flatnonzero(labels == c)  # object indices in cluster c
            if len(members) == 0:
                continue  # empty cluster: keep the old medoid
            # TODO 2 (update): new_medoids[c] = the member with the smallest
            #                  sum of distances to all other members
            raise NotImplementedError("k_medoids: update step")

        if np.array_equal(new_medoids, medoids):
            break  # nothing changed: converged
        medoids = new_medoids
    return labels, medoids

k-medoids only finds a *local* optimum that depends on the random start. The given helper
`best_k_medoids` runs it with several seeds and keeps the result with the lowest cost.

In [ ]:
def medoid_cost(D: np.ndarray, labels: np.ndarray, medoids: np.ndarray) -> float:
    """Sum of distances of every object to the medoid of its cluster."""
    return float(D[np.arange(len(labels)), medoids[labels]].sum())

def best_k_medoids(D: np.ndarray, k: int, n_init: int = 10, seed: int = 0):
    """Run k_medoids with several seeds and keep the result with the lowest cost."""
    best = None
    for run in range(n_init):
        labels, medoids = k_medoids(D, k, seed=seed + run)
        cost = medoid_cost(D, labels, medoids)
        if best is None or cost < best[2]:
            best = (labels, medoids, cost)
    return best

In [ ]:
# Check Task 2
def test_k_medoids_blobs():
    pts, blob_truth = make_blobs(n_samples=150, centers=3, cluster_std=0.6, random_state=0)
    D_blobs = squareform(pdist(pts))
    labels, medoids = k_medoids(D_blobs, 3, seed=0)
    labels, medoids = np.asarray(labels), np.asarray(medoids)
    assert labels.shape == (150,), "one label per object"
    assert len(set(medoids.tolist())) == 3, "we need k = 3 different medoids"
    assert set(labels.tolist()) <= {0, 1, 2}, "labels must be 0, 1 or 2"
    for c in range(3):
        assert labels[medoids[c]] == c, f"the medoid of cluster {c} must belong to cluster {c}"
    own = D_blobs[np.arange(150), medoids[labels]]
    assert np.allclose(own, D_blobs[:, medoids].min(axis=1)), "every object must belong to its CLOSEST medoid"
    best_labels, _, _ = best_k_medoids(D_blobs, 3, n_init=5)
    ari = adjusted_rand_score(blob_truth, best_labels)
    assert ari > 0.999, f"three well separated blobs must be found exactly (ARI {ari:.3f})"


def test_k_medoids_vs_random():
    _, _, cost = best_k_medoids(D, K, n_init=5)
    rng = np.random.default_rng(1)
    random_costs = [D[:, rng.choice(len(D), K, replace=False)].min(axis=1).sum() for _ in range(20)]
    assert cost <= min(random_costs), \
        f"cost {cost:.2f} is worse than the best of 20 random medoid sets ({min(random_costs):.2f})"


ddm.check("k_medoids on three blobs", test_k_medoids_blobs)
ddm.check("k_medoids on our D beats random medoids", test_k_medoids_vs_random)

Now on our data. The medoids are real objects: we can look at them.

In [ ]:
try:
    labels_kmed, medoids, cost = best_k_medoids(D, K, n_init=10)
    print(f"k-medoids with k = {K}: cost {cost:.2f}, cluster sizes {np.bincount(labels_kmed).tolist()}")
    print("The medoids (one representative object per cluster):")
    display(df.loc[data.index[medoids]])
    print(pd.crosstab(labels_kmed, truth, rownames=["cluster"], colnames=[LABEL]))
except NotImplementedError as err:
    labels_kmed = None
    print(f"NOTICE: k_medoids is not implemented yet ({err}).")

## Task 3 · Optional alternative: k-means with our own distance

Lloyd's algorithm on the feature matrix $X$: assign every object to the closest centroid,
move every centroid to the **mean** of its members, repeat. Objective (for Euclidean distance):

$$\text{SSE} = \sum_{i=1}^{n} \lVert x_i - \mu_{c(i)} \rVert^2 .$$

The mean is the best centre only for the (squared) Euclidean distance. With the Manhattan
distance the best centre would be the median: try both and compare.

`euclidean` and `manhattan` below are short versions of our `minkowski` from 02, so that this
notebook works on its own.

In [ ]:
def k_means(
    X: np.ndarray,
    k: int,
    distance: Callable[[np.ndarray, np.ndarray], float],
    seed: int = 0,
    max_iter: int = 100,
):
    """Lloyd's k-means, with your own distance function in the assignment step.

    Parameters
    ----------
    X : array of shape (n, m)
        Numeric feature matrix (e.g. the output of preprocess).
    k : number of clusters
    distance : function ``distance(x, y) -> float``
    seed : seed for the random start centroids
    max_iter : maximum number of assignment/update rounds

    Returns
    -------
    labels : int array of shape (n,), values 0 .. k-1
    centroids : array of shape (k, m)
    """
    X = np.asarray(X, dtype=float)
    n = X.shape[0]
    rng = np.random.default_rng(seed)
    centroids = X[rng.choice(n, size=k, replace=False)].copy()  # start: k random objects
    labels = np.full(n, -1)

    for _ in range(max_iter):
        new_labels = np.empty(n, dtype=int)
        for i in range(n):
            # TODO 1 (assignment): distance(X[i], centroids[c]) for every c, keep the index of the smallest
            raise NotImplementedError("k_means: assignment step")

        if np.array_equal(new_labels, labels):
            break  # nothing changed: converged
        labels = new_labels

        for c in range(k):
            members = X[labels == c]
            if len(members) > 0:  # empty cluster: keep the old centroid
                # TODO 2 (update): centroids[c] = mean of the members (axis=0)
                raise NotImplementedError("k_means: update step")
    return labels, centroids

In [ ]:
def euclidean(x: np.ndarray, y: np.ndarray) -> float:
    return float(np.sqrt(((x - y) ** 2).sum()))


def manhattan(x: np.ndarray, y: np.ndarray) -> float:
    return float(np.abs(x - y).sum())


def sse(X: np.ndarray, labels: np.ndarray, centroids: np.ndarray) -> float:
    """Sum of squared Euclidean distances of every object to its centroid."""
    return float(((np.asarray(X) - np.asarray(centroids)[labels]) ** 2).sum())

def best_k_means(X: np.ndarray, k: int, distance, n_init: int = 10, seed: int = 0):
    """Run k_means with several seeds and keep the result with the lowest SSE."""
    best = None
    for run in range(n_init):
        labels, centroids = k_means(X, k, distance, seed=seed + run)
        cost = sse(X, labels, centroids)
        if best is None or cost < best[2]:
            best = (labels, centroids, cost)
    return best

In [ ]:
# Check Task 3 (compares with sklearn.cluster.KMeans)
def test_k_means():
    pts, blob_truth = make_blobs(n_samples=150, centers=3, cluster_std=0.6, random_state=0)
    labels, centroids = k_means(pts, 3, euclidean, seed=0)
    assert np.shape(labels) == (150,) and np.shape(centroids) == (3, 2), "labels (n,) and centroids (k, m)"
    best_labels, _, best_sse = best_k_means(pts, 3, euclidean, n_init=10)
    ari = adjusted_rand_score(blob_truth, best_labels)
    assert ari > 0.999, f"three well separated blobs must be found exactly (ARI {ari:.3f})"
    sk = KMeans(n_clusters=3, n_init=10, random_state=0).fit(pts)
    assert best_sse <= sk.inertia_ * 1.001, f"SSE {best_sse:.2f} is worse than scikit-learn ({sk.inertia_:.2f})"


ddm.check("k_means", test_k_means)

In [ ]:
labels_kmeans = {}
try:
    for name, dist_fn in [("Euclidean", euclidean), ("Manhattan", manhattan)]:
        lab, _, cost = best_k_means(X, K, dist_fn, n_init=10)
        labels_kmeans[f"k-means, {name} (ours)"] = lab
        print(f"k-means with {name:9s} distance: SSE {cost:.1f}, sizes {np.bincount(lab).tolist()}, "
              f"ARI vs {LABEL} {adjusted_rand_score(truth, lab):.3f}")
    print(f"scikit-learn KMeans SSE (inertia):  {KMeans(n_clusters=K, n_init=10, random_state=0).fit(X).inertia_:.1f}")
except NotImplementedError as err:
    print(f"NOTICE: k_means is not implemented yet ({err}). That is fine, Task 3 is optional.")

## Compare: our clusterings, the libraries, the known groups (given)

The Adjusted Rand Index (ARI) compares two labelings: 1 = identical partition, about 0 = no
better than chance. Label *names* do not matter, only who is grouped with whom.

In [ ]:
clusterings = {}
clusterings[f"{LINKAGE} linkage ({'ours' if source.startswith('our') else 'SciPy'}, k={K})"] = labels_agg
if labels_kmed is not None:
    clusterings[f"k-medoids (k={K})"] = labels_kmed
for name, lab in labels_kmeans.items():
    clusterings[f"{name} (k={K})"] = lab

library = {
    f"SciPy {LINKAGE} linkage (k={K})": fcluster(scipy_linkage(squareform(D, checks=False), method=LINKAGE), K, "maxclust") - 1,
    f"scikit-learn KMeans on X (k={K})": KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(X),
}

rows = []
for name, lab in {**clusterings, **library}.items():
    row = {"clustering": name, "sizes": np.bincount(lab).tolist(), f"ARI vs {LABEL}": adjusted_rand_score(truth, lab)}
    for lib_name, lib_lab in library.items():
        row[f"ARI vs {lib_name}"] = adjusted_rand_score(lib_lab, lab)
    rows.append(row)
comparison = pd.DataFrame(rows).set_index("clustering")
comparison.round(3)

In [ ]:
to_plot = {f"{LABEL} (known groups)": truth, **clusterings}
ddm.plot_clusters_pca(X, to_plot, title=f"{DATASET}: PCA of the feature matrix X")
plt.show()
ddm.plot_clusters_mds(D, to_plot, title=f"{DATASET}: MDS of the distance matrix D")
plt.show()

## Save the clusterings for 04_beyond (given)

In [ ]:
path = ddm.save_json(DATASET, "labels", {"rows": meta["rows"], "labelings": clusterings})
print(f"Saved {len(clusterings)} clusterings to {path}: {list(clusterings)}")

## Reflection (2 minutes)

1. Compare the cluster sizes of the three linkages. Single linkage tends to "chain": do we see
   that on our data? If not, what separates the single-linkage clusters instead?
2. k-medoids vs. k-means on X: which one agrees better with the known groups? Does that make
   it the *better* clustering?
3. Look at the medoids: are they typical objects of their cluster?